In [ ]:
import random
import torch.nn as nn
from typing import Callable
from network_encoding import (
    OpFactory, Node, Edge, DAG,
    CellSpec, CellNode, CellEdge, NetworkDAG,
)

# OpFactory = Callable[[int, int], nn.Module]

# ── operazioni NB301 ───────────────────────────────────────────────────────────

def _make_op(fn, name):
    fn.op_name = name
    return fn

def op_sep_conv3x3() -> OpFactory:
    def f(Ci, Co):
        return nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(Ci, Ci, 3, padding=1, groups=Ci, bias=False),
            nn.Conv2d(Ci, Co, 1, bias=False), nn.BatchNorm2d(Co),
            nn.ReLU(inplace=False),
            nn.Conv2d(Co, Co, 3, padding=1, groups=Co, bias=False),
            nn.Conv2d(Co, Co, 1, bias=False), nn.BatchNorm2d(Co),
        )
    return _make_op(f, "sep_conv3x3")

def op_sep_conv5x5() -> OpFactory:
    def f(Ci, Co):
        return nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(Ci, Ci, 5, padding=2, groups=Ci, bias=False),
            nn.Conv2d(Ci, Co, 1, bias=False), nn.BatchNorm2d(Co),
            nn.ReLU(inplace=False),
            nn.Conv2d(Co, Co, 5, padding=2, groups=Co, bias=False),
            nn.Conv2d(Co, Co, 1, bias=False), nn.BatchNorm2d(Co),
        )
    return _make_op(f, "sep_conv5x5")

def op_dil_conv3x3() -> OpFactory:
    def f(Ci, Co):
        return nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(Ci, Ci, 3, padding=2, dilation=2, groups=Ci, bias=False),
            nn.Conv2d(Ci, Co, 1, bias=False), nn.BatchNorm2d(Co),
        )
    return _make_op(f, "dil_conv3x3")

def op_dil_conv5x5() -> OpFactory:
    def f(Ci, Co):
        return nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(Ci, Ci, 5, padding=4, dilation=2, groups=Ci, bias=False),
            nn.Conv2d(Ci, Co, 1, bias=False), nn.BatchNorm2d(Co),
        )
    return _make_op(f, "dil_conv5x5")

def op_max_pool3x3() -> OpFactory:
    def f(Ci, Co): return nn.MaxPool2d(3, stride=1, padding=1)
    return _make_op(f, "max_pool3x3")

def op_avg_pool3x3() -> OpFactory:
    def f(Ci, Co): return nn.AvgPool2d(3, stride=1, padding=1)
    return _make_op(f, "avg_pool3x3")

def op_skip() -> OpFactory:
    def f(Ci, Co): return nn.Identity()
    return _make_op(f, "skip")

NB301_OPS: list[OpFactory] = [
    op_sep_conv3x3(), op_sep_conv5x5(),
    op_dil_conv3x3(), op_dil_conv5x5(),
    op_max_pool3x3(), op_avg_pool3x3(),
    op_skip(),
]

# ── topologia interna di una cella DARTS ──────────────────────────────────────
#
# Nodi:  ck2, ck1          → 2 input (c_{k-2}, c_{k-1})
#        n0, n1, n2, n3    → 4 nodi intermedi, ognuno riceve 2 archi
#        out               → output, concat di n0..n3
#
# Archi ricercabili (14 totali, 2 per nodo intermedio):
#   n0 ← sceglie 2 tra {ck2, ck1}
#   n1 ← sceglie 2 tra {ck2, ck1, n0}
#   n2 ← sceglie 2 tra {ck2, ck1, n0, n1}
#   n3 ← sceglie 2 tra {ck2, ck1, n0, n1, n2}
#
# Archi fissi (4 totali): n0,n1,n2,n3 → out con Identity

# per ogni nodo intermedio, i candidati src tra cui pescare 2
_CANDIDATES = {
    "n0": ["ck2", "ck1"],
    "n1": ["ck2", "ck1", "n0"],
    "n2": ["ck2", "ck1", "n0", "n1"],
    "n3": ["ck2", "ck1", "n0", "n1", "n2"],
}

def _make_darts_dag(edges_14: list[tuple[str, str, OpFactory]]) -> DAG:
    """
    Costruisce il DAG interno di una cella DARTS.
    edges_14: lista di 14 tuple (src, dst, op_factory) per gli archi ricercabili,
              nell'ordine n0(×2), n1(×2), n2(×2), n3(×2).
              Le 4 righe verso 'out' sono aggiunte automaticamente.
    """
    assert len(edges_14) == 8, f"servono 8 archi ricercabili (2 per nodo), ricevuti {len(edges_14)}"

    def identity_factory(Ci, Co): return nn.Identity()
    identity_factory.op_name = "identity"

    nodes = [
        Node("ck2"),
        Node("ck1"),
        Node("n0",  aggregation="sum"),
        Node("n1",  aggregation="sum"),
        Node("n2",  aggregation="sum"),
        Node("n3",  aggregation="sum"),
        Node("out", aggregation="concat"),
    ]
    edges = (
        [Edge(s, d, op) for s, d, op in edges_14] +
        [Edge("n0", "out", identity_factory),
         Edge("n1", "out", identity_factory),
         Edge("n2", "out", identity_factory),
         Edge("n3", "out", identity_factory)]
    )
    return DAG(nodes, edges, inputs=["ck2", "ck1"], outputs=["out"])


def sample_nb301_networks(
    N: int,
    C: int = 36,
    seed: int | None = None,
) -> list[NetworkDAG]:
    """
    Genera N reti NB301/DARTS casuali.

    Struttura fissa della rete (CIFAR-10):
      20 celle totali: normal(×6) → reduction → normal(×6) → reduction → normal(×6)
      (reduction alle posizioni 1/3 e 2/3)

    Ogni rete ha due CellSpec campionate indipendentemente:
      - normal_spec  : stessa topologia per tutte le 18 normal cell
      - reduction_spec: stessa topologia per le 2 reduction cell

    Le celle sono collegate nel NetworkDAG tramite CellEdge:
      ogni cella i riceve da cella[i-1] (c_{k-1}) e cella[i-2] (c_{k-2}),
      tranne le prime due che ricevono entrambe dallo stem.
    """
    rng = random.Random(seed)
    networks = []

    for _ in range(N):

        # ── campiona normal spec ───────────────────────────────────────────────
        normal_edges = []
        for dst, candidates in _CANDIDATES.items():
            srcs = rng.sample(candidates, 2)   # 2 src distinti
            for src in srcs:
                op = rng.choice(NB301_OPS)
                normal_edges.append((src, dst, op))

        # ── campiona reduction spec ────────────────────────────────────────────
        reduction_edges = []
        for dst, candidates in _CANDIDATES.items():
            srcs = rng.sample(candidates, 2)
            for src in srcs:
                op = rng.choice(NB301_OPS)
                reduction_edges.append((src, dst, op))

        normal_spec    = CellSpec("normal",    _make_darts_dag(normal_edges))
        reduction_spec = CellSpec("reduction", _make_darts_dag(reduction_edges))

        # ── costruisce le 20 celle ─────────────────────────────────────────────
        # posizioni reduction: dopo la 6a e la 12a cella (indici 6 e 13)
        # layout: N×6 R N×6 R N×6
        layout = []
        for i in range(20):
            if i == 6 or i == 13:
                layout.append(("r", reduction_spec))
            else:
                layout.append(("n", normal_spec))

        cell_nodes = [
            CellNode(f"c{i}", spec, C_in=C, C_out=C)
            for i, (_, spec) in enumerate(layout)
        ]

        # ── collega le celle: ogni cella riceve da k-1 e k-2 ─────────────────
        cell_edges = []
        for i in range(1, len(cell_nodes)):
            cell_edges.append(CellEdge(src=f"c{i-1}", dst=f"c{i}"))   # c_{k-1}
            if i >= 2:
                cell_edges.append(CellEdge(src=f"c{i-2}", dst=f"c{i}"))  # c_{k-2}

        networks.append(NetworkDAG(
            cell_nodes=cell_nodes,
            cell_edges=cell_edges,
            inputs=["c0"],
            outputs=["c19"],
        ))

    return networks


# ── stampa ────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    reti = sample_nb301_networks(N=2, seed=42)
    for i, net in enumerate(reti):
        print(f"\n── rete {i} ──")
        print(f"  {net}")
        # mostra solo la normal spec della prima rete
        if i == 0:
            normal = net.cell_nodes[0].cell_spec
            print(f"  normal spec — archi ricercabili:")
            for e in normal.dag.edges:
                if e.dst != "out":
                    print(f"    {e.src}→{e.dst} : {e.op.op_name}")
            print(f"  celle e tipi:")
            for cn in net.cell_nodes:
                print(f"    {cn}")


── rete 0 ──
  NetworkDAG(20 celle)
  normal spec — archi ricercabili:
    ck2→n0 : avg_pool3x3
    ck1→n0 : dil_conv3x3
    ck2→n1 : sep_conv5x5
    n0→n1 : avg_pool3x3
    ck2→n2 : avg_pool3x3
    n0→n2 : max_pool3x3
    ck2→n3 : sep_conv3x3
    n1→n3 : sep_conv3x3
  celle e tipi:
    CellNode('c0', 'normal', 36→36)
    CellNode('c1', 'normal', 36→36)
    CellNode('c2', 'normal', 36→36)
    CellNode('c3', 'normal', 36→36)
    CellNode('c4', 'normal', 36→36)
    CellNode('c5', 'normal', 36→36)
    CellNode('c6', 'reduction', 36→36)
    CellNode('c7', 'normal', 36→36)
    CellNode('c8', 'normal', 36→36)
    CellNode('c9', 'normal', 36→36)
    CellNode('c10', 'normal', 36→36)
    CellNode('c11', 'normal', 36→36)
    CellNode('c12', 'normal', 36→36)
    CellNode('c13', 'reduction', 36→36)
    CellNode('c14', 'normal', 36→36)
    CellNode('c15', 'normal', 36→36)
    CellNode('c16', 'normal', 36→36)
    CellNode('c17', 'normal', 36→36)
    CellNode('c18', 'normal', 36→36)
    CellNode('c1